In [ ]:
import matplotlib.pyplot as plt
import scipp as sc
from trex.instrument import Instrument
import scipp.constants as const
from scippneutron.tof import chopper_cascade

T_OFFSET = sc.scalar(1.7, unit="ms")
central_wavelength = sc.scalar(2.5, unit="Å")
rrm: int = 8  # repetition rate multiplication factor
mode = "High Flux"  # Chopper mode


trex = Instrument(wavelength=central_wavelength, rrm=rrm, mode=mode, t_offset=T_OFFSET)
res = trex.model.run()

# Squeeze the pulse dimension since we only have one pulse
events_at_sample = res["Monitor at Sample"].data.squeeze()
# Remove the events that don't make it to the detector
events_at_sample = events_at_sample[~events_at_sample.masks["blocked_by_others"]]

In [ ]:
fig, ax = plt.subplots()
# tof_sample = events_at_sample.hist(wavelength=200, tof=500).plot(norm='log', ax=ax)
toa_sample = events_at_sample.hist(wavelength=800, toa=1500).plot(
    norm="log", cbar=True, ax=ax, cmap="jet"
)
ax.set_title("TOA at sample position\n" + str(trex), fontsize=8)
# TOF line
line_tof = sc.linspace("tof", 70, 140, 100, unit="ms").to(unit="us")
line_wavelength_tof = (
    const.h / const.m_n / trex.monitors["Monitor at Sample"].distance * line_tof
).to(unit="Å")
ax.plot(line_tof, line_wavelength_tof, label="lambda=h/(m*L_sample)*TOF", alpha=0.2)

# # TOA line
line_toa = sc.linspace("toa", 70, 140, 100, unit="ms").to(unit="us")
line_wavelength_toa = (
    const.h
    / const.m_n
    / trex.monitors["Monitor at Sample"].distance
    * (line_toa - T_OFFSET.to(unit="us"))
).to(unit="Å")
ax.plot(
    line_toa, line_wavelength_toa, label="lambda=h/(m*L_sample)*(TOA-1.7 ms)", alpha=0.2
)

line_wavelength_toa = (
    const.h
    / const.m_n
    / trex.monitors["Monitor at Sample"].distance
    * (line_toa - sc.scalar(3.5, unit="ms").to(unit="us"))
).to(unit="Å")
ax.plot(
    line_toa, line_wavelength_toa, label="lambda=h/(m*L_sample)*(TOA-3.5 ms)", alpha=0.2
)

ax.legend()
ax.grid(alpha=0.6)
fig.tight_layout()

In [ ]:
frames = chopper_cascade.FrameSequence.from_source_pulse(
    time_min=sc.scalar(0.0, unit="ms"),
    time_max=sc.scalar(4.0, unit="ms"),  # ESS pulse is 3 ms, but it has a tail
    wavelength_min=sc.scalar(0.0, unit="angstrom"),
    wavelength_max=sc.scalar(4.0, unit="angstrom"),
)
frames = frames.chop(trex.chopper_cascade.values())
at_sample = frames.propagate_to(trex.monitors["Monitor at Sample"].distance)

fig, ax = at_sample.draw(transpose=True)
ax.set_title("Frame propagation\n" + str(trex), fontsize=9)
ax.legend()
fig.tight_layout()

In [ ]:
fig, ax = frames.acceptance_diagram()
ax.set_title("Chopper acceptance diagram\n" + str(trex), fontsize=9)
ax.legend()
fig.tight_layout()

In [ ]:
import scipp as sc

sc.constants.neutron_mass

In [ ]:
from scippneutron.tof.chopper_cascade import FrameSequence
from dataclasses import dataclass


@dataclass
class SubframeVertex:
    distance: sc.Variable
    time: sc.Variable
    wavelength: sc.Variable

    @property
    def birth_time(self) -> sc.Variable:
        v = sc.constants.h / (sc.constants.neutron_mass * self.wavelength)
        travel_time = (self.distance / v).to(unit=self.time.unit)
        return self.time - travel_time

def acceptance_paths(
    *,
    frames: FrameSequence,
    time_unit: str = 'ms',
    wavelength_unit: str = 'angstrom',
) -> dict[str, dict]:
    frame_paths = {}
    for i, frame in enumerate(frames):
        subframe_paths = []
        for isub, subframe in enumerate(frame.subframes):
            vertex = SubframeVertex(
                distance=frame.distance,
                time = subframe.time.to(unit=time_unit, copy=False),
                wavelength = subframe.wavelength.to(unit=wavelength_unit, copy=False)
            )
            subframe_paths.append(vertex)
        frame_paths[f"{frame.distance:c}"] = subframe_paths
    return frame_paths


In [ ]:
display(acceptance_paths(frames=frames).keys())
subframe_vertexes = acceptance_paths(frames=frames)
display(subframe_vertexes['162.005 m'])
subframe_vertexes['162.005 m'][0].birth_time

In [ ]:
import tof

source = tof.Source(facility='ess', neutrons=50_000_000)
display(source)
source.data

In [ ]:
import numpy as np

def _get_coord_or_midpoints(da: sc.DataArray, coord_name: str) -> sc.Variable:
    coord = da.coords[coord_name]
    if coord.ndim == 1 and da.coords.is_edges(coord_name):
        return sc.midpoints(coord)
    else:
        return coord

def get_points(da: sc.DataArray, *, xcoord_name: str = 'wavelength', ycoord_name: str = 'birth_time') -> np.ndarray:
    x_coord = _get_coord_or_midpoints(da, xcoord_name)
    y_coord = _get_coord_or_midpoints(da, ycoord_name)
    sizes = {**x_coord.sizes, **y_coord.sizes}
    x_coord = x_coord.broadcast(sizes=sizes).copy(deep=True)
    y_coord = y_coord.broadcast(sizes=sizes).copy(deep=True)
    # Overwriting unit to make a stack using scipp operator...
    x_coord.unit = 'dimensionless'
    y_coord.unit = 'dimensionless'
    xy_stack = sc.concat([x_coord, y_coord], dim='i').flatten(dims=sizes.keys(), to='pos').transpose()
    return xy_stack.values


In [ ]:
example_vertex = subframe_vertexes['162.005 m']
example_vertex

In [ ]:
from matplotlib.path import Path 
import numpy as np

def apply_mask(*, source: tof.Source, vertexes: list[SubframeVertex]) -> tof.Source:
    from copy import copy

    source = copy(source)
    if len(vertexes) == 0:
        return source

    da = source.data
    wav_time_points = get_points(da)
    if len(vertexes) < 1 :
        wavelengths = [vertexes[0].wavelength]
        birth_times = [vertexes[0].birth_time.to(unit=da.coords['birth_time'].unit)]
    else:
        dim = vertexes[0].wavelength.dim
        wavelengths = [vertex.wavelength for vertex in vertexes]
        birth_times = [vertex.birth_time.to(unit=da.coords['birth_time'].unit) for vertex in vertexes]

    inside = sc.ones_like(da.data).astype(bool)
    inside.unit = None
    inside = ~inside
    for birth_time, wavelength in zip(birth_times, wavelengths):
        vx = wavelength.values
        vy = birth_time.values
        verts = np.column_stack([vx, vy])
        path = Path(verts)
        sizes = da.sizes
        dims = sizes.keys()
        inside = inside | sc.array(
            dims=dims,
            values=path.contains_points(wav_time_points).reshape(tuple(sizes.values())),
        )

    source._data = da.assign_masks({f"{vertexes[0].distance:c}": ~inside})
    return source


masked_source = apply_mask(source=source, vertexes=example_vertex)
masked_source.data

In [ ]:
da = masked_source.data.squeeze()
mask_name = '162.005 m'
da[~da.masks[mask_name]].hist(birth_time=100, wavelength=100).plot()

In [ ]:
da[~da.masks[mask_name]].hist(birth_time=100, wavelength=100).plot(title=f'Events who survive at {mask_name}').save('event_selection.png')

In [ ]:
# from matplotlib.path import Path
# import numpy as np

# frame=frames[-1].propagate_to(trex.source.distance)


# for subframe in frame.subframes:
#     x = subframe.time
#     y = subframe.wavelength
#     x_unit = x.unit
#     y_unit = y.unit
#     # x_max = max(x_max, x.max().value)
#     # y_max = max(y_max, y.max().value)

#     poly_path = Path(
#         np.stack((x.values, y.values), axis=1)
#     )